In [4]:
import json
import ijson
import faiss
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from pathlib import Path

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

Config 

In [5]:
CHUNKS_PATH = Path("../data/chunks/corpus_chunks_v2_e5.json")
OUTPUT_DIR = Path("../data/embeddings/e5_base")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "intfloat/multilingual-e5-base"

In [37]:
BATCH_SIZE = 16 
MAX_LENGTH = 512
SHARD_SIZE = 10_000
MAX_RECORDS = None  
TEXT_COL = "raw_text"

Load Model

In [38]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("device:", device)
print("dtype:", dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
).to(device)

model.eval()

device: cuda
dtype: torch.float16


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=Tru

Hàm encode chuẩn cho E5

In [39]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state.float() * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

In [40]:
@torch.no_grad()
def encode_texts(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            outputs = model(**inputs)

        embeddings = mean_pooling(
            outputs.last_hidden_state,
            inputs["attention_mask"],
        )

        embeddings = F.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(
            embeddings.cpu().numpy().astype("float32")
        )

        del inputs, outputs, embeddings

        if device == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(all_embeddings)

In [41]:
def iter_chunks_json_array(path):
    with open(path, "rb") as f: 
        for item in ijson.items(f, "item"):
            yield item 

In [42]:
def save_shard(shard_id, embeddings, metadata_rows): 
    emb_path = OUTPUT_DIR / f"embeddings_part_{shard_id:05d}.npy"
    meta_path = OUTPUT_DIR / f"metadata_part_{shard_id:05d}.parquet"
    embeddings = np.asarray(embeddings, dtype="float32")
    metadata=pd.DataFrame(metadata_rows)
    
    np.save(emb_path,embeddings) 
    metadata.to_parquet(meta_path, index=False) 
    
    print("saved:", emb_path, embeddings.shape)
    print("saved:", meta_path, metadata.shape)

In [ ]:
batch_texts = []
batch_meta = []

shard_embeddings = []
shard_metadata = []

shard_id = 0
total_records = 0
total_embedded = 0

for item in tqdm(iter_chunks_json_array(CHUNKS_PATH)):
    if MAX_RECORDS is not None and total_records >= MAX_RECORDS:
        break
    
    total_records += 1 
    
    text = str(item.get(TEXT_COL, "")).strip()
    
    if not text:
        continue 
    
    batch_texts.append("passage: " + text)
    batch_meta.append({
        "doc_id": item.get("doc_id"),
        "chunk_id": item.get("chunk_id"),
        "url": item.get("url"),
        "topic": item.get("topic"),
        "text": text,
    })
    
    if len(batch_texts) >= BATCH_SIZE : 
        embeddings = encode_texts(batch_texts)
        shard_embeddings.append(embeddings)
        shard_metadata.extend(batch_meta)
        
        total_embedded += len(batch_texts) 
        batch_texts = []
        batch_meta = []
        
        if len(shard_metadata) >= SHARD_SIZE:
            shard_embeddings = np.vstack(shard_embeddings)

            save_shard(
                shard_id=shard_id,
                embeddings=shard_embeddings,
                metadata_rows=shard_metadata,
            )

            shard_id += 1
            shard_embeddings = []
            shard_metadata = []

if batch_texts:
    embeddings = encode_texts(batch_texts)

    shard_embeddings.append(embeddings)
    shard_metadata.extend(batch_meta)

    total_embedded += len(batch_texts)
    
if shard_metadata:
    shard_embeddings = np.vstack(shard_embeddings)

    save_shard(
        shard_id=shard_id,
        embeddings=shard_embeddings,
        metadata_rows=shard_metadata,
    )

print("total_records:", total_records)
print("total_embedded:", total_embedded)

10000it [02:06, 59.36it/s]

saved: ..\data\embeddings\e5_base\embeddings_part_00000.npy (10000, 768)
saved: ..\data\embeddings\e5_base\metadata_part_00000.parquet (10000, 5)


20000it [04:40, 28.78it/s] 

saved: ..\data\embeddings\e5_base\embeddings_part_00001.npy (10000, 768)
saved: ..\data\embeddings\e5_base\metadata_part_00001.parquet (10000, 5)


30000it [08:39, 28.98it/s]

...

250000it [1:15:13, 46.92it/s]

saved: ..\data\embeddings\e5_base\embeddings_part_00024.npy (10000, 768)
saved: ..\data\embeddings\e5_base\metadata_part_00024.parquet (10000, 5)


260000it [1:17:53, 48.61it/s]

saved: ..\data\embeddings\e5_base\embeddings_part_00025.npy (10000, 768)
saved: ..\data\embeddings\e5_base\metadata_part_00025.parquet (10000, 5)


261872it [1:18:20, 77.70it/s]

## Build FAISS Index

In [26]:
embedding_files = sorted(OUTPUT_DIR.glob("embeddings_part_*.npy"))
metadata_files = sorted(OUTPUT_DIR.glob("metadata_part_*.parquet"))

print("embedding shards:", len(embedding_files))
print("metadata shards:", len(metadata_files))

embedding shards: 1
metadata shards: 1


In [28]:
index = None 
all_metadata = []
offset = 0 

for emb_file, meta_file in tqdm(list(zip(embedding_files, metadata_files))):
    embeddings = np.load(emb_file).astype("float32")
    metadata = pd.read_parquet(meta_file)
    
    if index is None: 
        dim = embeddings.shape[1] 
        index = faiss.IndexFlatIP(dim)
        
    metadata.insert(
        0,
        "embedding_index",
        np.arange(offset, offset+len(metadata))
    )
    index.add(embeddings)
    all_metadata.append(metadata)

    offset += len(metadata)

metadata = pd.concat(all_metadata, ignore_index=True)
print("faiss vectors:", index.ntotal)
print("metadata rows:", len(metadata))

100%|██████████| 1/1 [00:00<00:00, 40.47it/s]

faiss vectors: 1000
metadata rows: 1000


In [29]:
INDEX_PATH = OUTPUT_DIR / "semantic_e5_base.faiss"
METADATA_PATH = OUTPUT_DIR / "chunks_metadata.parquet"
CONFIG_PATH = OUTPUT_DIR / "embedding_config.json"

In [30]:
faiss.write_index(index, str(INDEX_PATH))
metadata.to_parquet(METADATA_PATH, index=False)

In [31]:
config = {
    "model_name": MODEL_NAME,
    "text_column": TEXT_COL,
    "document_prefix": "passage: ",
    "query_prefix": "query: ",
    "max_length": MAX_LENGTH,
    "normalize_embeddings": True,
    "faiss_metric": "inner_product",
    "embedding_dim": int(index.d),
    "num_vectors": int(index.ntotal),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

config

{'model_name': 'intfloat/multilingual-e5-base',
 'text_column': 'raw_text',
 'document_prefix': 'passage: ',
 'query_prefix': 'query: ',
 'max_length': 512,
 'normalize_embeddings': True,
 'faiss_metric': 'inner_product',
 'embedding_dim': 768,
 'num_vectors': 1000}

## Seach thu 

In [32]:
index = faiss.read_index(str(INDEX_PATH))
metadata = pd.read_parquet(METADATA_PATH) 

In [33]:
def semantic_search(query, top_k = 5) : 
    query_text = "query: "  + query 
    
    query_embedding = encode_texts(
        [query_text],
        batch_size=1,
        max_length=MAX_LENGTH,
    )
    scores, ids = index.search(query_embedding, top_k)

    results = []
    
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue

        row = metadata.iloc[idx].to_dict()
        row["score"] = float(score)
        results.append(row)

    return pd.DataFrame(results)

In [36]:
semantic_search("cướp tiệm vàng", top_k=5)

,embedding_index,doc_id,chunk_id,url,topic,text,score
0,495,131,131_1,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,0.882593
1,494,131,131_0,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,0.874453
2,1,0,0_1,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",0.871581
3,649,176,176_0,https://kenh14.vn/bat-ngo-voi-danh-tinh-doi-tu...,Xã Hội,Bất ngờ với danh tính đối tượng dùng súng AK c...,0.870130
4,2,0,0_2,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",0.860817


In [35]:
semantic_search("quân đội Trung Quốc ở Thái Bình Dương", top_k=5)

,embedding_index,doc_id,chunk_id,url,topic,text,score
0,563,148,148_2,https://thanhnien.vn/chien-luoc-cua-my-o-indo-...,Thế giới,Chiến lược của Mỹ ở Indo-Pacific giữa sự trỗi ...,0.842808
1,561,148,148_0,https://thanhnien.vn/chien-luoc-cua-my-o-indo-...,Thế giới,Chiến lược của Mỹ ở Indo-Pacific giữa sự trỗi ...,0.838629
2,562,148,148_1,https://thanhnien.vn/chien-luoc-cua-my-o-indo-...,Thế giới,Chiến lược của Mỹ ở Indo-Pacific giữa sự trỗi ...,0.822338
3,572,153,153_1,https://tuoitre.vn/tin-the-gioi-1-8-dan-trung-...,Thế giới,Tin thế giới 1-8: Dân Trung Quốc phản đối nhà ...,0.817237
4,290,77,77_0,https://www.qdnd.vn/quoc-te/tin-tuc/my-nhat-ha...,Quốc tế,Mỹ-Nhật-Hàn tập trận phòng thủ tên lửa đạn đạo...,0.812130
